# Explainable and Adaptive Machine Learning for Fraud Risk Detection

**Reproducibility notebook** for *Explainable and Adaptive Machine Learning for Fraud Risk Detection in Bangladesh’s Digital Banking Ecosystem*.

- arXiv: **2610.39556**
- Author: **Md Najmus Swaqeeb**
- Presented at the **11th Annual Banking Conference 2026, BIBM, Dhaka, 28 October 2026**.

This public notebook is a cleaned, execution-oriented copy of the original research notebook. Recovery, Google Drive backup, model-reload, and archive-management cells have been removed; the experimental logic is preserved. Saved outputs are intentionally cleared so users can rerun the workflow.


## Important scope and data note

The experiments use **PaySim**, a public synthetic mobile-money fraud benchmark. They do **not** use real Bangladeshi banking transactions. The results therefore demonstrate a reproducible methodological framework and should not be interpreted as direct deployment performance for Bangladesh.

The PaySim CSV is not distributed in this repository. Obtain it from its authorized source and place it in `data/` using the filename shown below, or set the `PAYSIM_DATA` environment variable to its location.


In [ ]:
from pathlib import Path
import os
import random
import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_DIR = Path.cwd()
DATA_PATH = Path(os.environ.get(
    "PAYSIM_DATA",
    PROJECT_DIR / "data" / "PS_20174392719_1491204439457_log.csv"
))
RESULT_DIR = PROJECT_DIR / "results"
FIGURE_DIR = PROJECT_DIR / "figures"
RESULT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset path:", DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"PaySim CSV not found at {DATA_PATH}. Place the dataset there or set PAYSIM_DATA."
    )


## 1. Load and inspect PaySim

The full PaySim dataset is loaded, then restricted to `CASH_OUT` and `TRANSFER`, the transaction types containing the fraud cases used in the study.


In [ ]:
import pandas as pd
import numpy as np

file_path = DATA_PATH

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nFraud distribution:")
print(df["isFraud"].value_counts())

print("\nOverall fraud rate:")
print(df["isFraud"].mean())

df.head()


In [ ]:
# ============================================================
# CELL 5 — FILTER FRAUD-RELEVANT TRANSACTION TYPES
# ============================================================

df_sub = df[
    df["type"].isin(["CASH_OUT", "TRANSFER"])
].copy()

print("Subset shape:", df_sub.shape)

print("\nTransaction type distribution:")
print(df_sub["type"].value_counts())

print("\nFraud distribution by transaction type:")
print(
    pd.crosstab(
        df_sub["type"],
        df_sub["isFraud"]
    )
)

print("\nTotal fraud cases:", df_sub["isFraud"].sum())
print("Fraud rate:", df_sub["isFraud"].mean())


## 2. Chronological split and leakage-aware feature construction

The study uses time-ordered splits rather than a random split: training steps 1–520, validation 521–632, and test 633–743. Historical account features are constructed sequentially using preceding observations only. A stable sort preserves original ordering within identical time steps.


In [ ]:
# ============================================================
# CELL 6 — CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train = df_sub[df_sub["step"] <= 520].copy()

val = df_sub[
    (df_sub["step"] > 520) &
    (df_sub["step"] <= 632)
].copy()

test = df_sub[df_sub["step"] > 632].copy()

print("TRAIN:", train.shape)
print("Fraud cases:", train["isFraud"].sum())
print("Fraud rate:", train["isFraud"].mean())

print("\nVALIDATION:", val.shape)
print("Fraud cases:", val["isFraud"].sum())
print("Fraud rate:", val["isFraud"].mean())

print("\nTEST:", test.shape)
print("Fraud cases:", test["isFraud"].sum())
print("Fraud rate:", test["isFraud"].mean())

print("\nStep ranges:")
print("Train:", train["step"].min(), "to", train["step"].max())
print("Validation:", val["step"].min(), "to", val["step"].max())
print("Test:", test["step"].min(), "to", test["step"].max())


In [ ]:
# ============================================================
# CELL 7 — LEAKAGE-SAFE FEATURE ENGINEERING
# ============================================================

import numpy as np
import pandas as pd

# Recombine in chronological order for feature construction
df_all = pd.concat([train, val, test], axis=0).copy()

# Preserve chronological transaction order
df_all = df_all.sort_values("step", kind="stable").copy()

# ------------------------------------------------------------
# 1. TIME FEATURES
# ------------------------------------------------------------

df_all["hour"] = df_all["step"] % 24
df_all["day"] = df_all["step"] // 24

df_all["is_night"] = (
    df_all["hour"] < 6
).astype(int)

# ------------------------------------------------------------
# 2. TRANSACTION TYPE
# ------------------------------------------------------------

df_all["type_transfer"] = (
    df_all["type"] == "TRANSFER"
).astype(int)

# ------------------------------------------------------------
# 3. HISTORICAL ORIGIN ACCOUNT FEATURES
# Only previous transactions are used
# ------------------------------------------------------------

df_all["origin_prev_count"] = (
    df_all.groupby("nameOrig").cumcount()
)

origin_cumsum = (
    df_all.groupby("nameOrig")["amount"]
    .cumsum()
    - df_all["amount"]
)

df_all["origin_prev_avg_amount"] = (
    origin_cumsum /
    df_all["origin_prev_count"].replace(0, np.nan)
).fillna(0)

# ------------------------------------------------------------
# 4. HISTORICAL DESTINATION ACCOUNT FEATURES
# ------------------------------------------------------------

df_all["dest_prev_count"] = (
    df_all.groupby("nameDest").cumcount()
)

# ------------------------------------------------------------
# 5. AMOUNT RELATIVE TO PREVIOUS ORIGIN AVERAGE
# ------------------------------------------------------------

df_all["amount_to_origin_avg"] = (
    df_all["amount"] /
    df_all["origin_prev_avg_amount"].replace(0, np.nan)
)

df_all["amount_to_origin_avg"] = (
    df_all["amount_to_origin_avg"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
)

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

feature_cols = [
    "step",
    "hour",
    "day",
    "is_night",
    "amount",
    "type_transfer",
    "origin_prev_count",
    "dest_prev_count",
    "origin_prev_avg_amount",
    "amount_to_origin_avg"
]

print("Feature engineering completed.")

print("\nDataset shape:", df_all.shape)

print("\nMissing values:")
print(df_all[feature_cols].isnull().sum())

print("\nFraud cases:", df_all["isFraud"].sum())

print("\nFeature preview:")
print(
    df_all[feature_cols + ["isFraud"]]
    .head()
)


In [ ]:
# ============================================================
# RECREATE CHRONOLOGICAL SPLITS WITH ENGINEERED FEATURES
# ============================================================

train = df_all[df_all["step"] <= 520].copy()

val = df_all[
    (df_all["step"] >= 521) &
    (df_all["step"] <= 632)
].copy()

test = df_all[df_all["step"] >= 633].copy()

print("TRAIN:", train.shape)
print("VALIDATION:", val.shape)
print("TEST:", test.shape)

print("\nEngineered feature check:")
print(test[feature_cols].head())


In [ ]:
# ============================================================
# CELL 9 — CREATE MODEL INPUTS
# ============================================================

features = [
    "step",
    "hour",
    "day",
    "is_night",
    "amount",
    "type_transfer",
    "origin_prev_count",
    "dest_prev_count",
    "origin_prev_avg_amount",
    "amount_to_origin_avg"
]

# Training
X_train = train[features].copy()
y_train = train["isFraud"].copy()

# Validation
X_val = val[features].copy()
y_val = val["isFraud"].copy()

# Test
X_test = test[features].copy()
y_test = test["isFraud"].copy()

print("MODEL DATA CREATED")
print("=" * 50)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nFraud cases:")
print("Train:", y_train.sum())
print("Validation:", y_val.sum())
print("Test:", y_test.sum())

print("\nFeatures:")
print(features)


## 3. Model training

Four reported supervised models are evaluated: Logistic Regression, Random Forest, Clean XGBoost, and LightGBM. The tree models below use the sampled training set described in the original experiment; the sample keeps all fraud cases and samples 20 legitimate transactions per fraud case with seed 42.

An additional class-weighted XGBoost experiment existed in the working notebook but is not part of the final four-model comparison, so it is omitted here to keep this notebook aligned with the reported results.


In [ ]:
# ============================================================
# CELL 10 — LOGISTIC REGRESSION
# ============================================================

from sklearn.linear_model import LogisticRegression
import time

start_time = time.time()

log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

log_reg.fit(
    X_train,
    y_train
)

# Probabilities
log_val_prob = log_reg.predict_proba(X_val)[:, 1]
log_test_prob = log_reg.predict_proba(X_test)[:, 1]

elapsed = time.time() - start_time

print(
    f"Logistic Regression training completed in: "
    f"{elapsed / 60:.2f} minutes"
)


In [ ]:
# ============================================================
# CELL 12 — CREATE SAMPLED TRAINING DATA
# ============================================================

train_fraud = train[
    train["isFraud"] == 1
].copy()

train_legit = train[
    train["isFraud"] == 0
].copy()

N_LEGIT_PER_FRAUD = 20

legit_sample = train_legit.sample(
    n=len(train_fraud) * N_LEGIT_PER_FRAUD,
    random_state=42
)

train_sample = pd.concat(
    [train_fraud, legit_sample],
    axis=0
)

train_sample = train_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

X_train_sample = train_sample[
    features
].copy()

y_train_sample = train_sample[
    "isFraud"
].copy()

print("Original training data:")
print("Fraud:", y_train.sum())
print("Legitimate:", (y_train == 0).sum())

print("\nSampled training data:")
print("Shape:", X_train_sample.shape)
print("Fraud:", y_train_sample.sum())
print("Legitimate:", (y_train_sample == 0).sum())
print("Fraud rate:", y_train_sample.mean())


In [ ]:
# ============================================================
# CELL 13 — RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier
import time

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_sample,
    y_train_sample
)

rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

elapsed = time.time() - start_time

print(
    f"Random Forest training completed in: "
    f"{elapsed / 60:.2f} minutes"
)


In [ ]:
# ============================================================
# CELL 17 — CLEAN XGBOOST
# ============================================================

import time
from xgboost import XGBClassifier

start_time = time.time()

xgb_model_clean = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model_clean.fit(
    X_train_sample,
    y_train_sample
)

# Probabilities
xgb_clean_val_prob = xgb_model_clean.predict_proba(X_val)[:, 1]

xgb_clean_test_prob = xgb_model_clean.predict_proba(X_test)[:, 1]

elapsed = time.time() - start_time

print(
    f"Clean XGBoost training completed in: "
    f"{elapsed / 60:.2f} minutes"
)


In [ ]:
# ============================================================
# CELL 20 — LIGHTGBM TRAINING
# ============================================================

from lightgbm import LGBMClassifier
import time

start_time = time.time()

lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train_sample,
    y_train_sample
)

lgb_val_prob = lgb_model.predict_proba(X_val)[:, 1]
lgb_test_prob = lgb_model.predict_proba(X_test)[:, 1]

elapsed = time.time() - start_time

print(
    f"LightGBM training completed in: "
    f"{elapsed / 60:.2f} minutes"
)


## 4. Validation-only threshold selection and final test evaluation

Decision thresholds are selected **only on the validation set**, maximizing validation F1. Those fixed thresholds are then applied once to the test probabilities. This separation is important for avoiding test-set threshold tuning.


In [ ]:
# ============================================================
# CELL 22 — VALIDATION-ONLY THRESHOLD SEARCH
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

models_for_threshold = {
    "Logistic Regression": log_val_prob,
    "Random Forest": rf_val_prob,
    "Clean XGBoost": xgb_clean_val_prob,
    "LightGBM": lgb_val_prob
}

thresholds = np.arange(0.01, 1.00, 0.01)

threshold_results = []

for model_name, val_prob in models_for_threshold.items():

    best_result = None

    for threshold in thresholds:

        pred = (val_prob >= threshold).astype(int)

        precision = precision_score(
            y_val, pred, zero_division=0
        )

        recall = recall_score(
            y_val, pred, zero_division=0
        )

        f1 = f1_score(
            y_val, pred, zero_division=0
        )

        tn, fp, fn, tp = confusion_matrix(
            y_val, pred
        ).ravel()

        fpr = fp / (fp + tn)

        result = {
            "Model": model_name,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fpr": fpr,
            "false_positives": fp,
            "false_negatives": fn,
            "true_positives": tp
        }

        if best_result is None or result["f1"] > best_result["f1"]:
            best_result = result

    threshold_results.append(best_result)

threshold_results_df = pd.DataFrame(threshold_results)

print("BEST VALIDATION THRESHOLD BY F1")
print("=" * 60)

print(threshold_results_df.to_string(index=False))


In [ ]:
# ============================================================
# CELL 24 — FINAL TEST EVALUATION
# USING VALIDATION-SELECTED THRESHOLDS
# ============================================================

model_test_probabilities = {
    "Logistic Regression": log_test_prob,
    "Random Forest": rf_test_prob,
    "Clean XGBoost": xgb_clean_test_prob,
    "LightGBM": lgb_test_prob
}

final_results = []

for _, row in threshold_results_df.iterrows():

    model_name = row["Model"]
    threshold = row["threshold"]

    test_prob = model_test_probabilities[model_name]

    pred = (test_prob >= threshold).astype(int)

    pr_auc = average_precision_score(
        y_test,
        test_prob
    )

    roc_auc = roc_auc_score(
        y_test,
        test_prob
    )

    precision = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        pred
    ).ravel()

    fpr = fp / (fp + tn)

    final_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

final_results_df = pd.DataFrame(final_results)

print("FINAL TEST RESULTS — VALIDATION-SELECTED THRESHOLDS")
print("=" * 75)

print(final_results_df.to_string(index=False))


In [ ]:
# Save the final four-model test table
final_results_df.to_csv(RESULT_DIR / "final_master_results.csv", index=False)
final_results_df


## 5. Probability calibration

Platt scaling is fitted on validation predictions from Clean XGBoost. The test labels are not used to fit the calibrator. Calibration quality is evaluated with the Brier score, where lower is better. A calibrated decision threshold is again selected on validation data before final test evaluation.


In [ ]:
# ============================================================
# CELL 26 — UNCALIBRATED BRIER SCORES
# ============================================================

from sklearn.metrics import brier_score_loss

# Validation
brier_val_uncal = brier_score_loss(
    y_val,
    xgb_clean_val_prob
)

# Test
brier_test_uncal = brier_score_loss(
    y_test,
    xgb_clean_test_prob
)

print("UNCALIBRATED CLEAN XGBOOST — BRIER SCORES")
print("-" * 50)

print(f"Validation Brier Score: {brier_val_uncal:.8f}")
print(f"Test Brier Score:       {brier_test_uncal:.8f}")


In [ ]:
# ============================================================
# CELL 27 — PLATT CALIBRATION
# ============================================================

from sklearn.linear_model import LogisticRegression

# Reshape probabilities into 2D arrays
X_val_cal = xgb_clean_val_prob.reshape(-1, 1)
X_test_cal = xgb_clean_test_prob.reshape(-1, 1)

# Fit Platt scaling using validation data only
platt_calibrator = LogisticRegression(
    random_state=42,
    max_iter=1000
)

platt_calibrator.fit(
    X_val_cal,
    y_val
)

# Generate calibrated probabilities
xgb_cal_val_prob = platt_calibrator.predict_proba(
    X_val_cal
)[:, 1]

xgb_cal_test_prob = platt_calibrator.predict_proba(
    X_test_cal
)[:, 1]

print("Platt calibration completed successfully.")

print("\nProbability ranges:")
print(
    "Validation:",
    xgb_cal_val_prob.min(),
    "to",
    xgb_cal_val_prob.max()
)

print(
    "Test:",
    xgb_cal_test_prob.min(),
    "to",
    xgb_cal_test_prob.max()
)


In [ ]:
# ============================================================
# CELL 28 — BRIER SCORE COMPARISON
# ============================================================

from sklearn.metrics import brier_score_loss

brier_val_cal = brier_score_loss(
    y_val,
    xgb_cal_val_prob
)

brier_test_cal = brier_score_loss(
    y_test,
    xgb_cal_test_prob
)

print("BRIER SCORE COMPARISON")
print("=" * 50)

print(f"Validation — Uncalibrated: {brier_val_uncal:.8f}")
print(f"Validation — Calibrated:   {brier_val_cal:.8f}")

print()

print(f"Test — Uncalibrated: {brier_test_uncal:.8f}")
print(f"Test — Calibrated:   {brier_test_cal:.8f}")


In [ ]:
# ============================================================
# CELL 30 — CALIBRATED THRESHOLD SEARCH ON VALIDATION DATA
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Search across the actual calibrated probability range
thresholds = np.arange(0.001, 0.322, 0.001)

calibrated_threshold_results = []

for threshold in thresholds:

    pred = (xgb_cal_val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        pred
    ).ravel()

    fpr = fp / (fp + tn)

    calibrated_threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

calibrated_threshold_df = pd.DataFrame(
    calibrated_threshold_results
)

# Best threshold according to validation F1
best_calibrated_threshold = (
    calibrated_threshold_df
    .sort_values("f1", ascending=False)
    .iloc[0]
)

print("CALIBRATED XGBOOST — BEST VALIDATION THRESHOLD")
print("=" * 60)

print(best_calibrated_threshold)


In [ ]:
# ============================================================
# CELL 32 — FINAL CALIBRATED XGBOOST TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Fixed threshold selected from validation only
calibrated_threshold = float(
    best_calibrated_threshold["threshold"]
)

# Apply to untouched test probabilities
calibrated_pred = (
    xgb_cal_test_prob >= calibrated_threshold
).astype(int)

# Ranking metrics
cal_pr_auc = average_precision_score(
    y_test,
    xgb_cal_test_prob
)

cal_roc_auc = roc_auc_score(
    y_test,
    xgb_cal_test_prob
)

# Classification metrics
cal_precision = precision_score(
    y_test,
    calibrated_pred,
    zero_division=0
)

cal_recall = recall_score(
    y_test,
    calibrated_pred,
    zero_division=0
)

cal_f1 = f1_score(
    y_test,
    calibrated_pred,
    zero_division=0
)

cal_tn, cal_fp, cal_fn, cal_tp = confusion_matrix(
    y_test,
    calibrated_pred
).ravel()

cal_fpr = cal_fp / (cal_fp + cal_tn)

print("FINAL CALIBRATED XGBOOST — TEST RESULTS")
print("=" * 65)

print("Model: Platt-Calibrated Clean XGBoost")
print(f"Threshold: {calibrated_threshold:.6f}")

print(f"\nPR-AUC:    {cal_pr_auc:.6f}")
print(f"ROC-AUC:   {cal_roc_auc:.6f}")
print(f"Precision: {cal_precision:.6f}")
print(f"Recall:    {cal_recall:.6f}")
print(f"F1:        {cal_f1:.6f}")
print(f"FPR:       {cal_fpr:.6f}")

print("\nConfusion Matrix")
print(f"TN: {cal_tn:,}")
print(f"FP: {cal_fp:,}")
print(f"FN: {cal_fn:,}")
print(f"TP: {cal_tp:,}")


In [ ]:
calibration_results = pd.DataFrame([{
    "Validation_Brier_Uncalibrated": brier_val_uncal,
    "Validation_Brier_Calibrated": brier_val_cal,
    "Test_Brier_Uncalibrated": brier_test_uncal,
    "Test_Brier_Calibrated": brier_test_cal,
    "Calibrated_Threshold": calibrated_threshold
}])
calibration_results.to_csv(RESULT_DIR / "calibration_results.csv", index=False)
calibration_results


## 6. Explainability with SHAP

SHAP is computed on a reproducible random sample of up to 5,000 test transactions (`random_state=42`). The analysis reports global mean absolute SHAP importance and a local explanation for a selected fraud case.

**Interpretation caution:** the study found strong importance for time-related variables. Because PaySim is synthetic and its fraud prevalence changes over time, these patterns require validation on appropriately protected real-world transaction data before operational use.


In [ ]:
# ============================================================
# CELL 42 — SETUP SHAP
# ============================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

print("SHAP version:", shap.__version__)
print("SHAP setup completed successfully.")


In [ ]:
# ============================================================
# CELL 43 — PREPARE DATA FOR SHAP ANALYSIS
# ============================================================

SHAP_SAMPLE_SIZE = min(5000, len(X_test))

X_shap = X_test.sample(
    n=SHAP_SAMPLE_SIZE,
    random_state=42
)

print("SHAP sample prepared successfully.")
print("Sample shape:", X_shap.shape)
print("Features:", X_shap.columns.tolist())


In [ ]:
# ============================================================
# CELL 44 — COMPUTE SHAP VALUES
# ============================================================

print("Creating SHAP TreeExplainer...")

explainer = shap.TreeExplainer(
    xgb_model_clean
)

print("Computing SHAP values...")

shap_values = explainer.shap_values(
    X_shap
)

# Handle possible list output
if isinstance(shap_values, list):
    shap_values_plot = shap_values[1]
else:
    shap_values_plot = shap_values

print("SHAP values computed successfully.")
print("SHAP values shape:", np.array(shap_values_plot).shape)


In [ ]:
# ============================================================
# CELL 45 — GLOBAL SHAP FEATURE IMPORTANCE
# ============================================================

# Mean absolute SHAP importance
shap_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Mean_Absolute_SHAP": np.abs(
        shap_values_plot
    ).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

print("GLOBAL SHAP FEATURE IMPORTANCE")
print("=" * 60)

print(shap_importance.to_string(index=False))

# Save results
shap_importance_path = os.path.join(
    str(RESULT_DIR),
    "shap_feature_importance.csv"
)

shap_importance.to_csv(
    shap_importance_path,
    index=False
)

print("\nSHAP feature importance saved:")
print(shap_importance_path)


In [ ]:
# ============================================================
# CELL 46 — SHAP SUMMARY PLOT
# ============================================================

plt.figure()

shap.summary_plot(
    shap_values_plot,
    X_shap,
    show=False
)

plt.tight_layout()

summary_path = os.path.join(
    str(FIGURE_DIR),
    "shap_summary_plot.png"
)

plt.savefig(
    summary_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("SHAP summary plot saved successfully:")
print(summary_path)


In [ ]:
# ============================================================
# CELL 47 — SHAP GLOBAL FEATURE IMPORTANCE BAR PLOT
# ============================================================

import matplotlib.pyplot as plt
import shap
import os

plt.figure()

shap.summary_plot(
    shap_values_plot,
    X_shap,
    plot_type="bar",
    show=False
)

plt.tight_layout()

bar_path = os.path.join(
    str(FIGURE_DIR),
    "shap_feature_importance_bar.png"
)

plt.savefig(
    bar_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("SHAP feature importance bar plot saved successfully:")
print(bar_path)


In [ ]:
# ============================================================
# CELL 48 — LOCAL SHAP EXPLANATION FOR A FRAUD CASE
# ============================================================

import numpy as np
import shap
import matplotlib.pyplot as plt
import os

# Find SHAP sample rows that are actual fraud cases
fraud_indices = X_shap.index[
    y_test.loc[X_shap.index].values == 1
]

print("Fraud cases available in SHAP sample:", len(fraud_indices))

if len(fraud_indices) == 0:
    print("No fraud case found in the SHAP sample.")
else:
    # Select the fraud case with the highest predicted fraud probability
    fraud_probs = xgb_model_clean.predict_proba(
        X_shap.loc[fraud_indices]
    )[:, 1]

    selected_index = fraud_indices[
        np.argmax(fraud_probs)
    ]

    selected_position = (
        X_shap.index.get_loc(selected_index)
    )

    print("Selected fraud transaction index:", selected_index)
    print(
        "Predicted fraud probability:",
        fraud_probs.max()
    )

    # Create explanation object
    explanation = shap.Explanation(
        values=shap_values_plot[selected_position],
        base_values=explainer.expected_value,
        data=X_shap.loc[selected_index].values,
        feature_names=feature_cols
    )

    # Waterfall plot
    plt.figure()

    shap.plots.waterfall(
        explanation,
        max_display=10,
        show=False
    )

    waterfall_path = os.path.join(
        project_path,
        "figures",
        "shap_local_fraud_explanation.png"
    )

    plt.savefig(
        waterfall_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("\nLocal SHAP explanation saved successfully:")
    print(waterfall_path)


In [ ]:
# ============================================================
# CELL 49 — SAVE XAI / SHAP RESULTS SUMMARY
# ============================================================

import pandas as pd
import os

# Top features from the global SHAP analysis
top_features = shap_importance.head(10).copy()

# Add rank
top_features.insert(
    0,
    "Rank",
    range(1, len(top_features) + 1)
)

# Save global SHAP importance
xai_results_path = os.path.join(
    str(RESULT_DIR),
    "xai_shap_results.csv"
)

top_features.to_csv(
    xai_results_path,
    index=False
)

# Save local explanation metadata
local_explanation_summary = pd.DataFrame([{
    "Selected_Transaction_Index": selected_index,
    "True_Label": 1,
    "Predicted_Fraud_Probability": float(fraud_probs.max()),
    "SHAP_Sample_Size": len(X_shap),
    "Fraud_Cases_In_SHAP_Sample": len(fraud_indices)
}])

local_results_path = os.path.join(
    str(RESULT_DIR),
    "shap_local_explanation_metadata.csv"
)

local_explanation_summary.to_csv(
    local_results_path,
    index=False
)

print("XAI / SHAP RESULTS SAVED SUCCESSFULLY")
print("=" * 65)

print("\nGlobal SHAP importance:")
print(xai_results_path)

print("\nLocal explanation metadata:")
print(local_results_path)

print("\nTop 5 important features:")
print(
    top_features.head(5).to_string(index=False)
)


## 7. Adaptive time-window risk prioritization

The adaptive analysis ranks the already-computed Clean XGBoost test probabilities within 24-step windows and flags approximately the highest-risk 1% in each window. This is an **operational prioritization analysis on the test period**, not model fitting and not the validation-based selection of the fixed classifier threshold.

It should therefore be interpreted as a workload/recall trade-off analysis: fewer false-positive alerts are obtained at the cost of substantially lower fraud recall.


In [ ]:
# ============================================================
# CELL 52 — ADAPTIVE TIME-WINDOW RISK SCORING
# ============================================================

import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# IMPORTANT:
# Use the already-trained Clean XGBoost model.
# No retraining is performed.
# ------------------------------------------------------------

# Test probabilities from the existing Clean XGBoost model
adaptive_test_prob = xgb_clean_test_prob.copy()

# Test dataframe aligned with X_test
adaptive_test = X_test.copy()

adaptive_test["fraud_probability"] = adaptive_test_prob
adaptive_test["actual_fraud"] = np.asarray(y_test)

# ------------------------------------------------------------
# Create chronological windows
# ------------------------------------------------------------

# Use original test step information
if "step" in adaptive_test.columns:
    adaptive_test["time_step"] = adaptive_test["step"].values
else:
    adaptive_test["time_step"] = df_all.loc[
        X_test.index, "step"
    ].values

# Create 24-step windows
WINDOW_SIZE = 24

adaptive_test["time_window"] = (
    adaptive_test["time_step"] // WINDOW_SIZE
)

# ------------------------------------------------------------
# Adaptive threshold:
# Select approximately the highest-risk 1% of transactions
# within each time window.
# ------------------------------------------------------------

def adaptive_threshold(group):
    return group["fraud_probability"].quantile(0.99)

window_thresholds = (
    adaptive_test
    .groupby("time_window")
    .apply(adaptive_threshold)
    .rename("adaptive_threshold")
    .reset_index()
)

adaptive_test = adaptive_test.merge(
    window_thresholds,
    on="time_window",
    how="left"
)

# Adaptive prediction
adaptive_test["adaptive_prediction"] = (
    adaptive_test["fraud_probability"]
    >= adaptive_test["adaptive_threshold"]
).astype(int)

print("Adaptive risk scoring completed.")

print("\nNumber of time windows:")
print(adaptive_test["time_window"].nunique())

print("\nAverage adaptive threshold:")
print(
    adaptive_test["adaptive_threshold"].mean()
)

print("\nAdaptive predictions:")
print(
    adaptive_test["adaptive_prediction"].value_counts()
)


In [ ]:
# ============================================================
# CELL 53 — ADAPTIVE RISK SCORING EVALUATION
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

adaptive_pred = adaptive_test[
    "adaptive_prediction"
].values

adaptive_true = adaptive_test[
    "actual_fraud"
].values

adaptive_precision = precision_score(
    adaptive_true,
    adaptive_pred,
    zero_division=0
)

adaptive_recall = recall_score(
    adaptive_true,
    adaptive_pred,
    zero_division=0
)

adaptive_f1 = f1_score(
    adaptive_true,
    adaptive_pred,
    zero_division=0
)

adaptive_tn, adaptive_fp, adaptive_fn, adaptive_tp = (
    confusion_matrix(
        adaptive_true,
        adaptive_pred
    ).ravel()
)

adaptive_fpr = (
    adaptive_fp /
    (adaptive_fp + adaptive_tn)
)

print("ADAPTIVE RISK SCORING — TEST RESULTS")
print("=" * 60)

print(f"Precision: {adaptive_precision:.6f}")
print(f"Recall:    {adaptive_recall:.6f}")
print(f"F1-score:  {adaptive_f1:.6f}")
print(f"FPR:       {adaptive_fpr:.6f}")

print("\nConfusion Matrix")
print(f"TN: {adaptive_tn:,}")
print(f"FP: {adaptive_fp:,}")
print(f"FN: {adaptive_fn:,}")
print(f"TP: {adaptive_tp:,}")


In [ ]:
adaptive_results = pd.DataFrame([{
    "Method": "Adaptive 1% Time-Window Risk Scoring",
    "Window_Size": WINDOW_SIZE,
    "Precision": adaptive_precision, "Recall": adaptive_recall,
    "F1": adaptive_f1, "FPR": adaptive_fpr,
    "TN": adaptive_tn, "FP": adaptive_fp, "FN": adaptive_fn, "TP": adaptive_tp
}])
adaptive_results.to_csv(RESULT_DIR / "adaptive_risk_results.csv", index=False)
adaptive_results


## 8. Reproducibility summary

Expected headline results from the reported experiment include Clean XGBoost PR-AUC ≈ 0.566 and ROC-AUC ≈ 0.907 at the validation-selected fixed threshold of 0.96, with 527 true positives and 192 false positives. Platt calibration reduces the test Brier score from ≈ 0.114 to ≈ 0.025. The adaptive top-1%-per-window policy reduces false positives from 192 to 117 while recall falls from ≈ 42% to ≈ 21%.

Exact results can depend on library versions. Record the environment used for each rerun and compare generated CSV files with the repository's reported result tables.


In [ ]:
import sys, platform, sklearn, xgboost, lightgbm, shap
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "lightgbm": lightgbm.__version__,
    "shap": shap.__version__,
    "random_seed": RANDOM_SEED,
}
environment


## Limitations

This notebook reproduces experiments on a synthetic benchmark. It does not establish deployment performance in Bangladeshi banks. Future validation should use appropriately protected real transaction data from Bangladeshi financial institutions across multiple periods, with particular attention to temporal distribution shift, false-alert workload, calibration, and whether time-related feature importance persists in real activity.
